# EA-TTM — Final LaTeX Report Assembly and Compilation

**Project:** Extreme-Aware Tiny Time Mixers (EA-TTM) for Retail Demand Forecasting under Regime Shifts  
**Phase:** 13 — Complete final report  
**Candidate:** Kamile Seidu  
**Student ID:** 22424181

This phase assembles the complete project report from the locked architecture, methodology, Phase 12 Results and Discussion, limitations, and final figures.

It produces:

- a complete LaTeX source folder;
- a compiled PDF;
- a ZIP containing the source and included figures;
- a compilation log;
- a rendered-page verification package.

No model training, tuning, reselection, or test re-evaluation is performed.

## Report governance

The model-development and final-evaluation locks remain binding:

$$
\text{Post-test tuning} = \text{False}.
$$

The report therefore documents the experiment as executed. Results that weaken the strongest version of the hypothesis, including the better MSE performance of the ungated cross-feed ablation, are retained rather than removed or rewritten.

In [1]:
from __future__ import annotations

import hashlib
import json
import os
import shutil
import subprocess
import textwrap
from pathlib import Path
from typing import Any

import pandas as pd

## Project preflight

In [2]:
from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/EA_TTM_RETAIL")
METADATA_DIR = PROJECT_ROOT / "metadata"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_TABLE_DIR = RESULTS_DIR / "tables"
RESULTS_FIGURE_DIR = RESULTS_DIR / "figures"
DELIVERABLES_DIR = PROJECT_ROOT / "deliverables"

REPORT_ROOT = DELIVERABLES_DIR / "EA_TTM_FINAL_REPORT"
SOURCE_DIR = REPORT_ROOT / "source"
FIGURE_DIR = SOURCE_DIR / "figures"
RENDER_DIR = REPORT_ROOT / "verification_renders"

for directory in (
    DELIVERABLES_DIR,
    REPORT_ROOT,
    SOURCE_DIR,
    FIGURE_DIR,
    RENDER_DIR,
):
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

required_artifacts = {
    "phase12_summary": METADATA_DIR / "12_RUN_SUMMARY.json",
    "phase10_summary": METADATA_DIR / "10_RUN_SUMMARY.json",
    "development_lock": METADATA_DIR / "09_MODEL_DEVELOPMENT_LOCK.json",
    "final_evaluation_lock": METADATA_DIR / "10_FINAL_EVALUATION_LOCK.json",
    "results_discussion_tex": RESULTS_DIR / "12_RESULTS_AND_DISCUSSION.tex",
    "final_report_results": RESULTS_TABLE_DIR / "12_FINAL_REPORT_RESULTS_TABLE.csv",
    "weekly_horizon": RESULTS_TABLE_DIR / "12_HORIZON_WEEKLY_SUMMARY.csv",
    "limitations": RESULTS_TABLE_DIR / "11_LIMITATIONS.csv",
    "model_complexity": RESULTS_TABLE_DIR / "11_MODEL_COMPLEXITY.csv",
    "fig_final_mse": RESULTS_FIGURE_DIR / "11_FIG01_FINAL_TEST_MSE.png",
    "fig_regime_mse": RESULTS_FIGURE_DIR / "11_FIG02_NORMAL_EXTREME_MSE.png",
    "fig_bootstrap": RESULTS_FIGURE_DIR / "11_FIG05_BOOTSTRAP_EA_VS_HP.png",
    "fig_focused_horizon": RESULTS_FIGURE_DIR / "11B_FIG07_FOCUSED_MSE_HP_EA_UNGATED.png",
    "fig_horizon_improvement": RESULTS_FIGURE_DIR / "11B_FIG04_PRIMARY_VS_HP_MSE_IMPROVEMENT.png",
    "fig_extreme_horizon": RESULTS_FIGURE_DIR / "11B_FIG05_PRIMARY_VS_HP_EXTREME_MSE_IMPROVEMENT.png",
    "fig_extreme_prevalence": RESULTS_FIGURE_DIR / "11B_FIG06_EXTREME_PREVALENCE_BY_HORIZON.png",
}

missing = [
    str(path)
    for path in required_artifacts.values()
    if not path.exists()
]

assert not missing, (
    "Phase 13 prerequisites are incomplete: "
    + "; ".join(missing)
)

Mounted at /content/drive


## Verify the locked experiment state

In [3]:
def load_json(path: Path) -> dict[str, Any]:
    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )

phase12 = load_json(
    METADATA_DIR / "12_RUN_SUMMARY.json"
)

phase10 = load_json(
    METADATA_DIR / "10_RUN_SUMMARY.json"
)

development_lock = load_json(
    METADATA_DIR / "09_MODEL_DEVELOPMENT_LOCK.json"
)

final_evaluation_lock = load_json(
    METADATA_DIR / "10_FINAL_EVALUATION_LOCK.json"
)

assert phase12["post_test_tuning_permitted"] is False
assert development_lock["post_lock_tuning_permitted"] is False
assert final_evaluation_lock["post_test_tuning_permitted"] is False
assert final_evaluation_lock["test_results_may_change_model_selection"] is False

PRIMARY_MODEL = development_lock["primary_model"]
PRIMARY_EPOCH = int(
    development_lock["primary_epoch"]
)

assert PRIMARY_MODEL == "Phase 08 best-balanced EA-TTM"
assert PRIMARY_EPOCH == 15

print("Primary model:", PRIMARY_MODEL)
print("Primary epoch:", PRIMARY_EPOCH)
print("Final-test interval:", final_evaluation_lock["final_test_interval"])
print("Post-test tuning permitted:", False)

Primary model: Phase 08 best-balanced EA-TTM
Primary epoch: 15
Final-test interval: ['d_1914', 'd_1941']
Post-test tuning permitted: False


## Prepare report source assets

In [4]:
# Clean only generated Phase 13 source contents.
for child in SOURCE_DIR.iterdir():
    if child.is_file():
        child.unlink()
    elif child.is_dir() and child.name != "figures":
        shutil.rmtree(child)

for child in FIGURE_DIR.iterdir():
    if child.is_file():
        child.unlink()

figure_sources = {
    "final_test_mse.png": required_artifacts["fig_final_mse"],
    "regime_mse.png": required_artifacts["fig_regime_mse"],
    "bootstrap.png": required_artifacts["fig_bootstrap"],
    "focused_horizon_mse.png": required_artifacts["fig_focused_horizon"],
    "horizon_mse_improvement.png": required_artifacts["fig_horizon_improvement"],
    "extreme_horizon_improvement.png": required_artifacts["fig_extreme_horizon"],
    "extreme_prevalence.png": required_artifacts["fig_extreme_prevalence"],
}

for destination_name, source_path in figure_sources.items():
    shutil.copy2(
        source_path,
        FIGURE_DIR / destination_name,
    )

shutil.copy2(
    RESULTS_DIR / "12_RESULTS_AND_DISCUSSION.tex",
    SOURCE_DIR / "results_discussion.tex",
)

print("Copied figures:", len(figure_sources))

Copied figures: 7


## Create final report tables

In [5]:
results_table = pd.read_csv(
    RESULTS_TABLE_DIR / "12_FINAL_REPORT_RESULTS_TABLE.csv"
)

weekly_horizon_table = pd.read_csv(
    RESULTS_TABLE_DIR / "12_HORIZON_WEEKLY_SUMMARY.csv"
)

limitations_table = pd.read_csv(
    RESULTS_TABLE_DIR / "11_LIMITATIONS.csv"
)

complexity_table = pd.read_csv(
    RESULTS_TABLE_DIR / "11_MODEL_COMPLEXITY.csv"
)

display_names = {
    "Weekly Seasonal Naive": "Seasonal Naive",
    "Zero-shot TTM": "Zero-shot TTM",
    "Frozen HP-TTM": "HP-TTM",
    "Retail-only gated adapter": "Retail-only gated",
    "Ungated cross-feed adapter": "Ungated cross-feed",
    "Primary EA-TTM": "Primary EA-TTM",
}

results_for_latex = results_table.copy()
results_for_latex["model"] = (
    results_for_latex["model"]
    .map(display_names)
    .fillna(results_for_latex["model"])
)

numeric_cols = [
    "mse",
    "rmse",
    "mae",
    "wape",
    "normal_mse",
    "extreme_mse",
    "positive_extreme_mse",
    "negative_extreme_mse",
]

for column in numeric_cols:
    results_for_latex[column] = (
        results_for_latex[column]
        .map(lambda value: f"{value:.4f}")
    )

# IMPORTANT:
# No caption/label here. Pandas then emits only a tabular environment.
# main.tex supplies the surrounding table float, caption and label.
results_latex = results_for_latex[
    [
        "model",
        "mse",
        "rmse",
        "mae",
        "wape",
        "normal_mse",
        "extreme_mse",
    ]
].to_latex(
    index=False,
    escape=True,
    column_format="lrrrrrr",
)

(SOURCE_DIR / "final_results_table.tex").write_text(
    results_latex,
    encoding="utf-8",
)

weekly_latex_frame = weekly_horizon_table[
    [
        "period",
        "mean_overall_mse_improvement_percent",
        "mean_extreme_mse_improvement_percent",
        "mean_positive_extreme_mse_improvement_percent",
        "mean_negative_extreme_mse_improvement_percent",
    ]
].copy()

weekly_latex_frame.columns = [
    "Period",
    "Overall MSE",
    "Extreme MSE",
    "Positive extreme",
    "Negative extreme",
]

for column in weekly_latex_frame.columns[1:]:
    weekly_latex_frame[column] = weekly_latex_frame[column].map(
        lambda value: f"{value:.2f}\\%"
    )

weekly_latex = weekly_latex_frame.to_latex(
    index=False,
    escape=False,
    column_format="lrrrr",
)

(SOURCE_DIR / "horizon_weekly_table.tex").write_text(
    weekly_latex,
    encoding="utf-8",
)

assert "\\begin{table}" not in results_latex
assert "\\begin{table}" not in weekly_latex
assert "\\begin{tabular}" in results_latex
assert "\\begin{tabular}" in weekly_latex

## Create bibliography

In [6]:
references_bib = r"""
@inproceedings{ekambaram2024ttm,
  title     = {Tiny Time Mixers (TTMs): Fast Pre-trained Models for Enhanced Zero/Few-Shot Forecasting of Multivariate Time Series},
  author    = {Ekambaram, Vijay and Jati, Arindam and Dayama, Pankaj and Mukherjee, Sumanta and Nguyen, Nam H. and Gifford, Wesley M. and Reddy, Chandra and Kalagnanam, Jayant},
  booktitle = {Advances in Neural Information Processing Systems},
  volume    = {37},
  year      = {2024}
}

@article{makridakis2022m5background,
  title   = {The M5 competition: Background, organization, and implementation},
  author  = {Makridakis, Spyros and Spiliotis, Evangelos and Assimakopoulos, Vassilios},
  journal = {International Journal of Forecasting},
  volume  = {38},
  number  = {4},
  pages   = {1325--1336},
  year    = {2022},
  doi     = {10.1016/j.ijforecast.2021.07.007}
}

@article{makridakis2022m5accuracy,
  title   = {M5 accuracy competition: Results, findings, and conclusions},
  author  = {Makridakis, Spyros and Spiliotis, Evangelos and Assimakopoulos, Vassilios},
  journal = {International Journal of Forecasting},
  volume  = {38},
  number  = {4},
  pages   = {1346--1364},
  year    = {2022},
  doi     = {10.1016/j.ijforecast.2021.11.013}
}
"""

(SOURCE_DIR / "references.bib").write_text(
    references_bib.strip() + "\\n",
    encoding="utf-8",
)

1226

## Create complete LaTeX report

In [7]:
main_tex = r"""
\documentclass[10pt,a4paper]{article}

\usepackage[margin=1.8cm]{geometry}
\usepackage{graphicx}
\usepackage{booktabs}
\usepackage{array}
\usepackage{amsmath,amssymb}
\usepackage{xcolor}
\usepackage{hyperref}
\usepackage{enumitem}
\usepackage{caption}
\usepackage{float}
\usepackage{tikz}
\usetikzlibrary{arrows.meta,positioning,shapes.geometric,fit}

\hypersetup{
  colorlinks=true,
  linkcolor=black,
  citecolor=black,
  urlcolor=black,
  pdftitle={Extreme-Aware Tiny Time Mixers for Retail Demand Forecasting under Regime Shifts},
  pdfauthor={Kamile Seidu}
}

\setlength{\parindent}{0pt}
\setlength{\parskip}{5pt}
\setlist[itemize]{leftmargin=*,nosep}
\setlist[enumerate]{leftmargin=*}

\title{\textbf{Extreme-Aware Tiny Time Mixers (EA-TTM) for Retail Demand Forecasting under Regime Shifts}}
\author{Kamile Seidu\\Student ID: 22424181}
\date{}

\begin{document}
\maketitle

\begin{abstract}
Retail demand forecasting is difficult because unit sales are intermittent and may change abruptly around events, price movements, and other local regime shifts. This project investigates whether a lightweight extreme-aware residual adapter can improve a frozen Tiny Time Mixer (TTM) while retaining the computational advantages of a small pre-trained time-series model. Using 30,490 bottom-level M5 store-item series, a pre-trained TTM was evaluated zero-shot, adapted by head probing, frozen, and then augmented with a 2,466-parameter EA-TTM adapter. The adapter combines recent robust demand statistics, horizon-aligned Slim TTM Decoder features, and future-known retail covariates. A learned Extreme Gate scales a signed residual correction. Model development was locked before opening the final 28-day test interval. The primary EA-TTM reduced locked-test MSE from 4.5630 for head-probed TTM to 4.5286, a 0.75\% improvement, and reduced extreme-regime MSE by 1.86\%. Positive-extreme MSE improved by 2.13\%, while MAE, WAPE, normal-regime MSE, and negative-extreme MSE deteriorated. Horizon analysis showed lower overall and extreme MSE on 27 of 28 forecast steps. The gate displayed meaningful extreme ranking, but an ungated cross-feed ablation obtained lower MSE than the gated primary model. The results therefore support lightweight decoder cross-feed and residual adaptation for large-error and positive-tail robustness, while showing that explicit multiplicative gating is not uniformly forecast-optimal.
\end{abstract}

\section{Introduction}

Pre-trained time-series models offer a way to transfer temporal representations across forecasting tasks without training a large model from scratch. Tiny Time Mixers (TTM) was developed as a compact pre-trained forecasting family designed for efficient zero-shot and few-shot transfer, including a frozen-backbone head-probing procedure \cite{ekambaram2024ttm}. Its small parameter footprint makes it especially attractive when compute is constrained.

Retail demand presents a difficult transfer setting. The M5 competition data contain intermittent unit-sales series together with calendar, event, SNAP, and price information \cite{makridakis2022m5background,makridakis2022m5accuracy}. At the store-item level, demand can contain long zero stretches, abrupt spikes, drops, and temporary changes in local regime. A model that performs well on average may therefore still make its most consequential errors during unusual demand periods.

This project asks:

\begin{quote}
\textit{Can a lightweight extreme-aware adapter improve TTM forecasting during retail demand shocks while preserving normal-period accuracy and computational efficiency?}
\end{quote}

The proposed EA-TTM keeps the retail-adapted TTM frozen and learns only a small residual adapter. The adapter receives recent-demand signals, horizon-specific representations extracted from the frozen Slim TTM Decoder, and future-known retail covariates. It produces both an extreme-routing probability and a signed correction to the frozen TTM forecast.

The study contributes a tightly controlled evaluation of this idea. The experiment uses a fixed validation split, a locked 28-day final test, explicit model-development and post-test locks, ablations that isolate decoder cross-feed and gating, and horizon-wise analysis across all 28 forecast steps.

\section{Background and Problem Setting}

\subsection{Tiny Time Mixers}

TTM is a lightweight pre-trained forecasting model based on mixer-style temporal processing. The published TTM design supports zero-shot forecasting and efficient target-domain head probing in which the backbone remains frozen while the decoder and forecasting head are adapted \cite{ekambaram2024ttm}. This project uses that multi-level adaptation principle as the starting point.

\subsection{Retail Demand Setting}

The experiment uses the M5 retail forecasting data, whose bottom level contains 30,490 store-item series. The dataset combines daily unit sales with calendar information, events, state-specific SNAP indicators, and weekly sell prices \cite{makridakis2022m5background}. This setting is suitable for investigating abrupt demand changes because the series are sparse and heterogeneous.

In the processed training observations used by this project, the zero-demand rate was approximately 68.37\%, highlighting the intermittency of bottom-level retail sales. The forecasting context length was fixed at 512 days and the forecast horizon at 28 days.

\subsection{Specific Research Gap}

The original TTM evaluation established strong transfer performance across standard time-series benchmarks but did not specifically establish robustness to abrupt retail demand shifts. The research problem in this project is therefore not to replace TTM, but to test whether a very small residual module can improve its behavior where forecast errors become unusually large.

\section{Methodology}

\subsection{Data Split and Experimental Governance}

The M5 bottom-level store-item series were used with the following fixed origins:

\begin{itemize}
    \item Training origins: 512, 960, 1409, and 1857.
    \item Validation origin: 1885, forecasting $d_{1886}$--$d_{1913}$.
    \item Locked final-test origin: 1913, forecasting $d_{1914}$--$d_{1941}$.
\end{itemize}

The training set therefore contained 121,960 windows and the validation and test sets each contained 30,490 series. The final-test target was not opened until all primary model choices and ablation definitions had been locked.

\subsection{Baseline Progression}

Three stages establish the forecasting base:

\begin{enumerate}
    \item \textbf{Weekly Seasonal Naive:} repeats the latest seven observed days across the 28-day horizon.
    \item \textbf{Zero-shot TTM:} applies the complete pre-trained TTM without retail-domain training.
    \item \textbf{Head-probed TTM (HP-TTM):} freezes the pre-trained backbone while training the Slim TTM Decoder and forecasting head on the retail training windows. The complete HP-TTM is then frozen.
\end{enumerate}

\subsection{Horizon-Aligned Decoder Cross-Feed}

Let the frozen decoder state be
\[
D \in \mathbb{R}^{B\times 1\times 9\times 128}.
\]
The frozen forecasting-head weights are reshaped by forecast horizon and patch. For each horizon $h$, a 128-dimensional decoder representation is formed as
\[
d_{b,h,k} = \sum_{p=1}^{9} W^{(h)}_{p,k}D_{b,p,k}.
\]
This alignment adds no trainable parameters and reconstructs the frozen head contribution to numerical precision.

\subsection{Retail Features}

The recent-demand stream uses a 28-day lookback and contains five features:

\begin{enumerate}
    \item recent median;
    \item robust MAD scale;
    \item latest seven-day mean minus the preceding seven-day mean;
    \item 28-day linear trend;
    \item zero-demand rate.
\end{enumerate}

The future-known stream contains eight horizon-specific features: state-specific SNAP indicator, event indicator, current price-record indicator, valid price-change indicator, standardized log price, standardized weekly log-price change, day-of-week sine, and day-of-week cosine. All standardization statistics are fitted only on training windows.

\subsection{EA-TTM Architecture}

The complete shock encoder input is
\[
u_h = [z_t;d_h;c_h] \in \mathbb{R}^{141},
\]
where $z_t\in\mathbb{R}^{5}$ is the recent-demand vector, $d_h\in\mathbb{R}^{128}$ is the frozen decoder cross-feed, and $c_h\in\mathbb{R}^{8}$ contains future-known covariates.

The Retail Shock Encoder is
\[
141 \rightarrow 16,
\]
implemented as a linear layer followed by GELU and LayerNorm.

The Extreme Gate is
\[
g_h = \sigma(W_g r_h+b_g),
\]
while the correction head is
\[
\Delta_h = f_{\Delta}(r_h),
\]
using a $16\rightarrow8\rightarrow1$ MLP. The final forecast is
\[
\hat{y}_{EA,h}
=
\hat{y}_{HP,h}
+
g_h\Delta_h.
\]

\begin{figure}[H]
\centering
\begin{tikzpicture}[
  node distance=7mm and 7mm,
  box/.style={draw, rounded corners, align=center, minimum height=9mm, minimum width=28mm, font=\small},
  smallbox/.style={draw, rounded corners, align=center, minimum height=8mm, minimum width=24mm, font=\scriptsize},
  arrow/.style={-{Latex[length=2mm]}, thick}
]
\node[box] (hist) {Historical\\Demand};
\node[box, right=of hist] (ttm) {Frozen\\HP-TTM};
\node[smallbox, below=of hist] (recent) {Recent Demand\\Signals (5)};
\node[smallbox, below=of ttm] (decoder) {Slim Decoder\\Features (128)};
\node[smallbox, right=of decoder] (future) {Future-known\\Covariates (8)};
\node[box, below=13mm of decoder] (encoder) {Retail Shock Encoder\\141 $\rightarrow$ 16};
\node[smallbox, below left=of encoder] (gate) {Extreme Gate\\$g_h$};
\node[smallbox, below right=of encoder] (corr) {Correction Head\\$\Delta_h$};
\node[box, right=18mm of gate] (final) {Final Forecast\\$\hat y_{HP,h}+g_h\Delta_h$};

\draw[arrow] (hist) -- (ttm);
\draw[arrow] (recent) -- (encoder);
\draw[arrow] (decoder) -- (encoder);
\draw[arrow] (future) -- (encoder);
\draw[arrow] (ttm) -- (decoder);
\draw[arrow] (encoder) -- (gate);
\draw[arrow] (encoder) -- (corr);
\draw[arrow] (gate) -- (final);
\draw[arrow] (corr) -- (final);
\draw[arrow] (ttm.east) to[out=0,in=90] (final.north);
\end{tikzpicture}
\caption{EA-TTM residual adaptation architecture. The HP-TTM is frozen; only the small retail adapter is trainable.}
\label{fig:architecture}
\end{figure}

The resulting adapter contains only 2,466 trainable parameters. Its final correction layer is initialized to zero, so the initial EA-TTM forecast is exactly identical to the frozen HP-TTM forecast.

\subsection{Extreme-Regime Definition}

For each forecast origin, the recent 28-day center is
\[
m_t=\mathrm{median}(y_{t-27:t}),
\]
and the robust scale is
\[
s_t=1.4826\,\mathrm{median}(|y_i-m_t|).
\]
The effective scale is
\[
\tilde{s}_t=\max(s_t,1).
\]
For future horizon $h$, the shock score is
\[
q_{t,h}
=
\frac{|y_{t+h}-m_t|}{\tilde{s}_t}.
\]
The extreme threshold is the training-only 95th percentile of these scores. In this experiment the locked threshold was $\tau=2.0$, giving a training extreme rate of approximately 4.71\%.

\subsection{Training Objective}

The primary adapter jointly optimizes forecasting and gate supervision. The forecast component is MSE on the final EA-TTM forecast:
\[
\mathcal{L}_{forecast}
=
\mathrm{MSE}(\hat Y_{EA},Y).
\]
The gate component is binary cross-entropy with logits:
\[
\mathcal{L}_{gate}
=
\mathrm{BCEWithLogits}(\ell,e).
\]
Both components are normalized by fixed training-reference quantities before summation. AdamW with learning rate $10^{-3}$, batch size 512, gradient clipping, learning-rate reduction on plateau, and early stopping were used. The primary checkpoint was selected by a balanced validation criterion that equally weights relative overall and extreme MSE.

\subsection{Ablations}

Two controlled ablations were trained before the final test:

\begin{itemize}
    \item \textbf{Retail-only gated adapter:} removes the 128-dimensional decoder cross-feed and uses only the five recent-demand features plus eight future-known covariates.
    \item \textbf{Ungated cross-feed adapter:} retains the full 141-dimensional input but removes the Extreme Gate and directly adds the learned residual correction.
\end{itemize}

These ablations isolate the incremental contribution of decoder cross-feed and multiplicative gating.

\subsection{Evaluation Metrics}

Overall evaluation uses MSE, RMSE, MAE, and WAPE. MSE is additionally decomposed into normal, all-extreme, positive-extreme, and negative-extreme regimes. Horizon-wise metrics are computed separately for each of the 28 forecast steps. A deterministic 2,000-replicate paired bootstrap over store-item series is used as a robustness diagnostic for the HP-TTM versus primary EA-TTM MSE difference.

\input{results_discussion.tex}

\begin{table}[H]
\centering
\small
\caption{Locked final-test forecasting performance. Lower values indicate lower forecast error.}
\label{tab:final-results}
\resizebox{\textwidth}{!}{\input{final_results_table.tex}}
\end{table}

\begin{figure}[H]
\centering
\includegraphics[width=0.90\textwidth]{figures/final_test_mse.png}
\caption{Overall locked final-test MSE across the fixed comparison set.}
\label{fig:overall-mse}
\end{figure}

\begin{figure}[H]
\centering
\includegraphics[width=0.90\textwidth]{figures/regime_mse.png}
\caption{Normal-regime and extreme-regime MSE on the locked final test.}
\label{fig:regime-mse}
\end{figure}

\begin{figure}[H]
\centering
\includegraphics[width=0.92\textwidth]{figures/focused_horizon_mse.png}
\caption{Horizon-wise MSE for frozen HP-TTM, primary EA-TTM, and the ungated cross-feed ablation.}
\label{fig:focused-horizon}
\end{figure}

\begin{figure}[H]
\centering
\includegraphics[width=0.92\textwidth]{figures/horizon_mse_improvement.png}
\caption{Primary EA-TTM percentage MSE improvement over HP-TTM across the 28-day horizon. Positive values indicate lower EA-TTM MSE.}
\label{fig:horizon-improvement}
\end{figure}

\begin{figure}[H]
\centering
\includegraphics[width=0.92\textwidth]{figures/extreme_horizon_improvement.png}
\caption{Primary EA-TTM extreme-MSE improvement over HP-TTM across the 28-day horizon.}
\label{fig:extreme-horizon}
\end{figure}

\begin{figure}[H]
\centering
\includegraphics[width=0.92\textwidth]{figures/extreme_prevalence.png}
\caption{Extreme, positive-extreme, and negative-extreme prevalence across the forecast horizon.}
\label{fig:extreme-prevalence}
\end{figure}

\begin{table}[H]
\centering
\small
\caption{Mean EA-TTM percentage improvement over HP-TTM across four seven-day horizon blocks.}
\label{tab:horizon-blocks}
\resizebox{\textwidth}{!}{\input{horizon_weekly_table.tex}}
\end{table}

\begin{figure}[H]
\centering
\includegraphics[width=0.75\textwidth]{figures/bootstrap.png}
\caption{Paired bootstrap interval for the series-level MSE difference HP-TTM minus primary EA-TTM.}
\label{fig:bootstrap}
\end{figure}

\section{Limitations}

Several limitations constrain the scope of the conclusions.

First, the empirical evaluation is based on one retail dataset and one locked 28-day final-test interval. The results therefore do not establish generalization to other retailers, product hierarchies, countries, or forecast horizons.

Second, the primary EA-TTM improves MSE but worsens MAE and WAPE relative to HP-TTM. Its main contribution is therefore concentrated in squared-error reduction, particularly large positive-tail errors, rather than universal accuracy improvement.

Third, negative extremes are rare and the primary EA-TTM worsens negative-extreme MSE. Evidence for downward demand shocks is consequently weaker than evidence for positive demand surges.

Fourth, the ungated cross-feed ablation outperforms the gated primary model on overall and extreme MSE. The gate learns meaningful extreme-regime information, but its multiplicative routing can suppress otherwise useful residual corrections.

Fifth, the shock definition is a robust statistical operationalization rather than a retailer-defined economic event taxonomy. Alternative shock definitions could change both the gate target and the measured effect.

Finally, the covariate set is intentionally compact to respect the compute-constrained design. Richer promotional, stock availability, weather, macroeconomic, and event information could provide additional signal.

\section{Conclusion}

This project examined whether a lightweight extreme-aware residual adapter could improve a frozen Tiny Time Mixer on intermittent retail demand. The experiment established a strong zero-shot TTM baseline, improved it through head probing, and then attached a 2,466-parameter adapter that combines recent robust demand statistics, frozen decoder representations, and future-known retail covariates.

The locked primary EA-TTM reduced overall final-test MSE by 0.75\% and extreme MSE by 1.86\% relative to HP-TTM. Positive-extreme MSE improved by 2.13\%, and overall and extreme MSE improved on 27 of 28 forecast horizons. The gate showed meaningful ranking and routed stronger corrections to extreme observations. However, MAE, WAPE, normal-regime MSE, and negative-extreme MSE deteriorated, and the ungated cross-feed ablation achieved still lower MSE.

The most defensible conclusion is therefore that lightweight decoder cross-feed and retail-aware residual adaptation can improve the large-error and positive-tail behavior of a frozen pre-trained forecasting model under tight parameter and compute constraints. The evidence does not establish that multiplicative gating is universally optimal.

\section{Future Work}

Future work should test the architecture on additional retail datasets and longer forecast periods, evaluate alternative extreme definitions, and explicitly address the scarcity of negative extreme events. A natural architectural extension is to replace the single multiplicative gate with a softer or multi-branch routing mechanism that can preserve useful residual corrections while still exposing an interpretable extreme score. Richer promotional, stock, weather, and economic covariates could also be incorporated. Finally, probabilistic forecasting would provide a stronger treatment of uncertainty around extreme demand than point forecasts alone.

\section*{Reproducibility Statement}

All major implementation stages were executed as separate audited phases. Dataset and split hashes, feature contracts, model checkpoints, model-development locks, and the final-evaluation lock were persisted before reporting. The final test covered $d_{1914}$--$d_{1941}$. The primary model was fixed as the Phase 08 best-balanced EA-TTM checkpoint at epoch 15 before the final-test target was opened, and post-test tuning was explicitly prohibited.

\bibliographystyle{plain}
\bibliography{references}

\end{document}
"""

(SOURCE_DIR / "main.tex").write_text(
    main_tex.strip() + "\\n",
    encoding="utf-8",
)

18709

## Install LaTeX and PDF verification dependencies

In [8]:
def command_exists(name: str) -> bool:
    return shutil.which(name) is not None

required_commands = {
    "pdflatex": "TeX Live",
    "bibtex": "TeX Live binaries",
    "pdftoppm": "poppler-utils",
    "pdfinfo": "poppler-utils",
}

missing_before = [
    command
    for command in required_commands
    if not command_exists(command)
]

if missing_before:
    env = os.environ.copy()
    env["DEBIAN_FRONTEND"] = "noninteractive"

    update_result = subprocess.run(
        ["apt-get", "update", "-qq"],
        check=False,
        capture_output=True,
        text=True,
        env=env,
    )

    if update_result.returncode != 0:
        raise RuntimeError(
            "apt-get update failed.\n"
            f"STDOUT:\n{update_result.stdout[-4000:]}\n"
            f"STDERR:\n{update_result.stderr[-4000:]}"
        )

    apt_packages = [
        "texlive-latex-base",
        "texlive-latex-recommended",
        "texlive-latex-extra",
        "texlive-fonts-recommended",
        "texlive-pictures",
        "poppler-utils",
    ]

    install_result = subprocess.run(
        [
            "apt-get",
            "install",
            "-y",
            "-qq",
            "--no-install-recommends",
            *apt_packages,
        ],
        check=False,
        capture_output=True,
        text=True,
        env=env,
    )

    if install_result.returncode != 0:
        raise RuntimeError(
            "LaTeX dependency installation failed.\n"
            f"Packages: {apt_packages}\n"
            f"STDOUT:\n{install_result.stdout[-6000:]}\n"
            f"STDERR:\n{install_result.stderr[-6000:]}"
        )

missing_after = [
    command
    for command in required_commands
    if not command_exists(command)
]

assert not missing_after, (
    "Required commands are still unavailable after installation: "
    + ", ".join(missing_after)
)

# Verify the actual LaTeX style files used by main.tex.
required_tex_files = [
    "geometry.sty",
    "graphicx.sty",
    "booktabs.sty",
    "array.sty",
    "amsmath.sty",
    "amssymb.sty",
    "xcolor.sty",
    "hyperref.sty",
    "enumitem.sty",
    "caption.sty",
    "float.sty",
    "tikz.sty",
]

missing_tex_files = []

for tex_file in required_tex_files:
    probe = subprocess.run(
        ["kpsewhich", tex_file],
        check=False,
        capture_output=True,
        text=True,
    )
    if probe.returncode != 0 or not probe.stdout.strip():
        missing_tex_files.append(tex_file)

assert not missing_tex_files, (
    "Required LaTeX style files are unavailable: "
    + ", ".join(missing_tex_files)
)

print(
    "LaTeX/PDF dependencies available:",
    ", ".join(required_commands),
)
print(
    "LaTeX style-file preflight: PASSED"
)

LaTeX/PDF dependencies available: pdflatex, bibtex, pdftoppm, pdfinfo
LaTeX style-file preflight: PASSED


## Compile the report

In [9]:
compile_log_path = REPORT_ROOT / "13_LATEX_COMPILE_LOG.txt"

# Remove stale compiler outputs from previous failed Phase 13 attempts.
for stale_name in [
    "main.aux",
    "main.bbl",
    "main.blg",
    "main.log",
    "main.out",
    "main.toc",
    "main.pdf",
]:
    stale_path = SOURCE_DIR / stale_name
    if stale_path.exists():
        stale_path.unlink()

main_tex_path = SOURCE_DIR / "main.tex"
generated_main_tex = main_tex_path.read_text(
    encoding="utf-8"
)

assert "\\usepackage{microtype}" not in generated_main_tex
assert "\\usepackage[T1]{fontenc}" not in generated_main_tex

print(
    "Font-expansion compatibility preflight: PASSED"
)

compile_commands = [
    ["pdflatex", "-interaction=nonstopmode", "-halt-on-error", "main.tex"],
    ["bibtex", "main"],
    ["pdflatex", "-interaction=nonstopmode", "-halt-on-error", "main.tex"],
    ["pdflatex", "-interaction=nonstopmode", "-halt-on-error", "main.tex"],
]

compile_log_parts = []

for command in compile_commands:
    result = subprocess.run(
        command,
        cwd=SOURCE_DIR,
        check=False,
        capture_output=True,
        text=True,
    )

    compile_log_parts.extend(
        [
            "$ " + " ".join(command),
            result.stdout,
            result.stderr,
            "",
        ]
    )

    compile_log_path.write_text(
        "\\n".join(compile_log_parts),
        encoding="utf-8",
    )

    if result.returncode != 0:
        diagnostic = (
            result.stdout[-8000:]
            + "\\n"
            + result.stderr[-4000:]
        )
        raise RuntimeError(
            "LaTeX compilation failed while running: "
            + " ".join(command)
            + "\\n\\nLast compiler output:\\n"
            + diagnostic
            + f"\\n\\nFull log: {compile_log_path}"
        )

compiled_pdf = SOURCE_DIR / "main.pdf"

assert compiled_pdf.exists()
assert compiled_pdf.stat().st_size > 50_000

FINAL_PDF_PATH = (
    DELIVERABLES_DIR
    / "EA_TTM_Final_Report_Kamile_Seidu.pdf"
)

shutil.copy2(
    compiled_pdf,
    FINAL_PDF_PATH,
)

print("Compiled PDF:", FINAL_PDF_PATH)

Font-expansion compatibility preflight: PASSED
Compiled PDF: /content/drive/MyDrive/EA_TTM_RETAIL/deliverables/EA_TTM_Final_Report_Kamile_Seidu.pdf


## Render and verify the compiled PDF

In [10]:
# Clear previous verification renders.
for existing in RENDER_DIR.glob("*"):
    if existing.is_file():
        existing.unlink()

render_prefix = RENDER_DIR / "page"

render_result = subprocess.run(
    [
        "pdftoppm",
        "-png",
        "-r",
        "120",
        str(FINAL_PDF_PATH),
        str(render_prefix),
    ],
    check=True,
    capture_output=True,
    text=True,
)

pdfinfo_result = subprocess.run(
    [
        "pdfinfo",
        str(FINAL_PDF_PATH),
    ],
    check=True,
    capture_output=True,
    text=True,
)

rendered_pages = sorted(
    RENDER_DIR.glob("page-*.png")
)

assert len(rendered_pages) >= 6

for page_path in rendered_pages:
    assert page_path.stat().st_size > 5_000

pdfinfo_text = pdfinfo_result.stdout

assert "Pages:" in pdfinfo_text
assert "Page size:" in pdfinfo_text

verification_report = {
    "pdf_path": str(FINAL_PDF_PATH),
    "pdf_size_bytes": int(
        FINAL_PDF_PATH.stat().st_size
    ),
    "rendered_page_count": len(
        rendered_pages
    ),
    "all_rendered_pages_nonempty": True,
    "pdfinfo": pdfinfo_text,
}

VERIFICATION_PATH = (
    REPORT_ROOT
    / "13_PDF_VERIFICATION.json"
)

VERIFICATION_PATH.write_text(
    json.dumps(
        verification_report,
        indent=2,
    ),
    encoding="utf-8",
)

print("Rendered pages:", len(rendered_pages))

Rendered pages: 10


## Create the LaTeX source ZIP

In [11]:
# Remove LaTeX temporary files from the source package.
temporary_suffixes = {
    ".aux",
    ".bbl",
    ".blg",
    ".log",
    ".out",
    ".toc",
}

for path in SOURCE_DIR.iterdir():
    if (
        path.is_file()
        and path.suffix in temporary_suffixes
    ):
        path.unlink()

# Keep the compiled PDF out of the source directory:
# the deliverable PDF is stored separately.
source_pdf = SOURCE_DIR / "main.pdf"
if source_pdf.exists():
    source_pdf.unlink()

ZIP_BASE = (
    DELIVERABLES_DIR
    / "EA_TTM_Final_Report_LaTeX_Source"
)

zip_created = shutil.make_archive(
    str(ZIP_BASE),
    "zip",
    root_dir=SOURCE_DIR,
)

FINAL_ZIP_PATH = Path(
    zip_created
)

assert FINAL_ZIP_PATH.exists()
assert FINAL_ZIP_PATH.stat().st_size > 100_000

print("Source ZIP:", FINAL_ZIP_PATH)

Source ZIP: /content/drive/MyDrive/EA_TTM_RETAIL/deliverables/EA_TTM_Final_Report_LaTeX_Source.zip


## Create Phase 13 manifest

In [12]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()

REPORT_MANIFEST_PATH = (
    METADATA_DIR
    / "13_FINAL_REPORT_MANIFEST.json"
)

report_manifest = {
    "status": "PHASE_13_FINAL_REPORT_COMPLETE",
    "primary_model": PRIMARY_MODEL,
    "primary_epoch": PRIMARY_EPOCH,
    "post_test_tuning_permitted": False,
    "final_pdf": str(
        FINAL_PDF_PATH
    ),
    "final_pdf_sha256": sha256_file(
        FINAL_PDF_PATH
    ),
    "latex_source_zip": str(
        FINAL_ZIP_PATH
    ),
    "latex_source_zip_sha256": sha256_file(
        FINAL_ZIP_PATH
    ),
    "compile_log": str(
        compile_log_path
    ),
    "verification": str(
        VERIFICATION_PATH
    ),
    "rendered_page_count": len(
        rendered_pages
    ),
    "source_directory": str(
        SOURCE_DIR
    ),
}

REPORT_MANIFEST_PATH.write_text(
    json.dumps(
        report_manifest,
        indent=2,
    ),
    encoding="utf-8",
)

status_path = PROJECT_ROOT / "PROJECT_STATUS.md"

status_lines = [
    "# EA-TTM Project Status",
    "",
    "## Current stage",
    "Phase 13 — Final LaTeX report compiled and verified.",
    "",
    "## Experimental governance",
    "- Model development: LOCKED",
    "- Final test: COMPLETE",
    "- Post-test tuning: PROHIBITED",
    "",
    "## Deliverables",
    f"- Final PDF: {FINAL_PDF_PATH}",
    f"- LaTeX source ZIP: {FINAL_ZIP_PATH}",
    "",
    "## Next phase",
    (
        "Phase 14 — Final defense presentation using the locked "
        "architecture, results, horizon graphs, ablations, and conclusions."
    ),
]

status_path.write_text(
    "\n".join(status_lines),
    encoding="utf-8",
)

565

## Return block

Return only this block after a successful run.

In [13]:
return_block = {
    "status": "PHASE_13_FINAL_REPORT_COMPLETE",
    "primary_model": PRIMARY_MODEL,
    "primary_epoch": PRIMARY_EPOCH,
    "post_test_tuning_permitted": False,
    "deliverables": {
        "final_pdf": str(
            FINAL_PDF_PATH
        ),
        "final_pdf_sha256": report_manifest[
            "final_pdf_sha256"
        ],
        "latex_source_zip": str(
            FINAL_ZIP_PATH
        ),
        "latex_source_zip_sha256": report_manifest[
            "latex_source_zip_sha256"
        ],
        "compile_log": str(
            compile_log_path
        ),
        "verification": str(
            VERIFICATION_PATH
        ),
        "rendered_page_count": len(
            rendered_pages
        ),
        "manifest": str(
            REPORT_MANIFEST_PATH
        ),
    },
}

print("PHASE 13 RETURN BLOCK")
print(
    json.dumps(
        return_block,
        indent=2,
    )
)

PHASE 13 RETURN BLOCK
{
  "status": "PHASE_13_FINAL_REPORT_COMPLETE",
  "primary_model": "Phase 08 best-balanced EA-TTM",
  "primary_epoch": 15,
  "post_test_tuning_permitted": false,
  "deliverables": {
    "final_pdf": "/content/drive/MyDrive/EA_TTM_RETAIL/deliverables/EA_TTM_Final_Report_Kamile_Seidu.pdf",
    "final_pdf_sha256": "1985f1ab264565cd2ac6b502a633944cde8d5488b20ccaa3b5b123a228616b13",
    "latex_source_zip": "/content/drive/MyDrive/EA_TTM_RETAIL/deliverables/EA_TTM_Final_Report_LaTeX_Source.zip",
    "latex_source_zip_sha256": "69926f5c72ffbb03146b26278bcb810c18f367dc8f171b117903c0a0d2298cd3",
    "compile_log": "/content/drive/MyDrive/EA_TTM_RETAIL/deliverables/EA_TTM_FINAL_REPORT/13_LATEX_COMPILE_LOG.txt",
    "verification": "/content/drive/MyDrive/EA_TTM_RETAIL/deliverables/EA_TTM_FINAL_REPORT/13_PDF_VERIFICATION.json",
    "rendered_page_count": 10,
    "manifest": "/content/drive/MyDrive/EA_TTM_RETAIL/metadata/13_FINAL_REPORT_MANIFEST.json"
  }
}
